# Closures and Decorators

If you have read any real Python code, you have seen lines that start with an `@`:

```python
@app.route("/users")
@property
@lru_cache
```

They are called **decorators**, and until now this course has deliberately avoided them. Chapter 8 even worked around one: when we needed to force a subclass to implement a method, we used `raise NotImplementedError` and noted that the proper tool involved decorators, which we had not covered yet.

This chapter covers them. But we are not going to start with the `@`. We are going to start with something you have already written — in chapter 7, without knowing what it was called — and build up until the `@` is the obvious next step.

**What we will learn:**

1. Functions as objects you can pass around and store
2. Nested functions and **closures**: how a function remembers values
3. Turning chapter 7's timer into a **decorator**, one step at a time
4. `*args` and `**kwargs`, so a decorator works on any function
5. `functools.wraps`, and the bug it fixes
6. The decorators you will actually meet: `@property`, `@staticmethod`, `@classmethod`, `@lru_cache`, `@abstractmethod`
7. Decorators that take arguments

---
# 1. Functions Are Objects

Chapter 7 introduced higher-order functions: functions that take or return other functions. That works because in Python a function is an ordinary **object**, like a string or a list.

You can check its type, look at its name, and put it in a variable:

In [1]:
def greet(name):
    return f"Hello, {name}!"

print(type(greet))
print(greet.__name__)

say_hello = greet                 # no parentheses -- we are naming the function, not calling it
print(say_hello("Shikhar"))

<class 'function'>
greet
Hello, Shikhar!


Notice `greet` and `greet()` are different things. **`greet` is the function itself; `greet()` runs it.** Nearly every decorator mistake comes back to that distinction.

Because functions are objects, they can go anywhere an object can — inside a list, or a dictionary:

In [2]:
def to_upper(text): return text.upper()
def to_title(text): return text.title()
def reverse(text):  return text[::-1]

formatters = {"upper": to_upper, "title": to_title, "reverse": reverse}

for name, func in formatters.items():
    print(f"{name:<8} -> {func('stranger things')}")

upper    -> STRANGER THINGS
title    -> Stranger Things
reverse  -> sgniht regnarts


That dictionary is a small dispatch table — a real pattern, and much cleaner than a long `if/elif` chain deciding which formatter to run.

---
# 2. Nested Functions and Closures

A function can be defined **inside** another function. The inner one is a local name, invisible from outside:

In [3]:
def outer():
    def inner():
        return "I am the inner function"
    return inner()          # note the parentheses: we call inner and return its result

print(outer())

try:
    inner()
except NameError as e:
    print("NameError:", e)

I am the inner function
NameError: name 'inner' is not defined


Now change one character — drop the parentheses — so the outer function returns the inner **function itself** rather than its result:

In [4]:
def outer():
    def inner():
        return "I am the inner function"
    return inner            # no parentheses: hand back the function

result = outer()
print(type(result))         # a function object, not a string
print(result.__name__)      # and it still knows its own name
print(result())             # now call it

<class 'function'>
inner
I am the inner function


## The Interesting Part

Here is where it stops being a curiosity. Watch what the inner function can see:

In [5]:
def create_adder(x):
    def adder(y):
        return x + y        # x belongs to create_adder, not to adder
    return adder

add_15 = create_adder(15)
add_100 = create_adder(100)

print(add_15(10))
print(add_100(10))

25
110


**You wrote this exact function in chapter 7**, as an example of a higher-order function. What we did not say then is that it has a name of its own.

Look carefully at what is happening. `create_adder(15)` has *already finished and returned* by the time we call `add_15(10)`. Its local variable `x` should be long gone. Yet `adder` still knows that `x` is 15.

A function that remembers variables from the scope it was defined in — even after that scope has finished — is called a **closure**. It "closes over" those variables.

You can see the captured value directly:

In [6]:
print(add_15.__closure__[0].cell_contents)
print(add_100.__closure__[0].cell_contents)

15
100


Two functions, built from the same code, each carrying its own remembered value. `add_15` and `add_100` are genuinely different objects.

**Why this matters:** a closure lets you build **customised functions on demand**. You write the logic once and stamp out configured versions of it.

In [7]:
def make_discount(percent):
    def apply(price):
        return round(price * (1 - percent / 100), 2)
    return apply

diwali_sale = make_discount(30)
clearance = make_discount(70)

print("₹2,499 in the Diwali sale:", diwali_sale(2499))
print("₹2,499 in clearance:      ", clearance(2499))

₹2,499 in the Diwali sale: 1749.3
₹2,499 in clearance:       749.7


## A Closure Trap Worth Knowing

A closure captures the **variable**, not the value it had at the time. That distinction is invisible until you build functions in a loop:

In [8]:
multipliers = []

for n in [2, 3, 4]:
    multipliers.append(lambda x: x * n)     # each lambda captures n itself

print([f(10) for f in multipliers])         # not [20, 30, 40]

[40, 40, 40]


All three functions share the same `n`, and by the time we call them the loop has finished with `n` at 4. So every one of them multiplies by 4.

The fix is to capture the *value* by making it a default argument, which is evaluated immediately when the function is defined:

In [9]:
multipliers = []

for n in [2, 3, 4]:
    multipliers.append(lambda x, n=n: x * n)    # n=n freezes the current value

print([f(10) for f in multipliers])

[20, 30, 40]


The function factory from earlier does not have this problem, because each call to `make_discount` creates a genuinely separate scope with its own `percent`. Building functions in a loop is the case to watch for.

## Changing a Captured Variable: `nonlocal`

A closure can *read* the enclosing variable freely. Trying to **assign** to it is a different matter — Python assumes any variable you assign to is local, so this fails:

In [10]:
def make_counter():
    count = 0
    def increment():
        count = count + 1       # Python treats count as local -- and it has no value yet
        return count
    return increment

counter = make_counter()
try:
    counter()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


The fix is the `nonlocal` keyword, which says "this name belongs to the enclosing function, not to me":

In [11]:
def make_counter():
    count = 0
    def increment():
        nonlocal count          # use the enclosing count, do not create a new local one
        count += 1
        return count
    return increment

page_views = make_counter()
print(page_views(), page_views(), page_views())

likes = make_counter()          # a separate counter with its own count
print(likes())

1 2 3
1


This is worth comparing with the `global` keyword from chapter 6:

| Keyword | Means |
|---|---|
| (nothing) | Create a new variable local to this function |
| `nonlocal` | Use the variable from the **enclosing function** |
| `global` | Use the variable at **module level** |

---
# 3. From Higher-Order Function to Decorator

Now we can build a decorator, and we will do it by improving something you already have.

In chapter 7 we wrote an AWS-style execution timer: a higher-order function that takes a customer's function, times it, and reports the duration for billing. Here it is again, trimmed slightly:

In [12]:
import time

def execution_timer(customer_function, data):
    start = time.perf_counter()
    customer_function(data)
    duration = time.perf_counter() - start
    print(f"AWS: billed for {duration:.1f} seconds")


def process_data(items):
    print(f"   processing {len(items)} items...")
    time.sleep(0.5)             # stand-in for real work


execution_timer(process_data, ["a", "b", "c"])

   processing 3 items...


AWS: billed for 0.5 seconds


It works, but it is awkward to live with:

- Everyone who calls `process_data` has to remember to write `execution_timer(process_data, data)` instead.
- The timer has to know how to call the function — here, that it takes exactly one argument.
- If you decide to stop timing, you edit every call site.

**Step 1: return a new function instead of calling it.**

Rather than running the customer's function immediately, we build a *replacement* for it — a function that times, calls, and reports — and hand that back:

In [13]:
def execution_timer(customer_function):
    def wrapper(data):
        start = time.perf_counter()
        customer_function(data)                 # the original function
        duration = time.perf_counter() - start
        print(f"AWS: billed for {duration:.1f} seconds")
    return wrapper                              # hand back the replacement


timed_process = execution_timer(process_data)
timed_process(["a", "b", "c"])

   processing 3 items...


AWS: billed for 0.5 seconds


`wrapper` is a closure: it remembers `customer_function` after `execution_timer` has returned.

**Step 2: reuse the original name.**

There is no rule that the replacement needs a new name. Assign it back over the original, and every existing call is timed with no other change:

In [14]:
def process_data(items):
    print(f"   processing {len(items)} items...")
    time.sleep(0.5)

process_data = execution_timer(process_data)    # replace the name with the timed version

process_data(["a", "b", "c"])                   # ordinary-looking call, now timed

   processing 3 items...


AWS: billed for 0.5 seconds


**Step 3: the `@` symbol.**

That line — `process_data = execution_timer(process_data)` — is common enough that Python gives it a shorthand. Write the decorator's name with an `@` directly above the `def`:

In [15]:
@execution_timer
def process_data(items):
    print(f"   processing {len(items)} items...")
    time.sleep(0.5)


process_data(["a", "b", "c"])

   processing 3 items...


AWS: billed for 0.5 seconds


That is the whole trick.

> ### **`@decorator` above a `def` means exactly `func = decorator(func)`.**

There is no other magic. A decorator is a function that takes a function and returns a replacement for it, and `@` is shorthand for the reassignment.

Read it that way and the `@app.route(...)` and `@property` lines in other people's code stop being mysterious.

## When Does a Decorator Actually Run?

One detail that clears up a lot of confusion: the decorator itself runs **once, when the function is defined** — not each time the function is called. Only the `wrapper` runs per call.

Put a print in both places and the order is obvious:

In [16]:
def noisy_decorator(func):
    print(f"   [decorating {func.__name__} -- happens once, at definition]")
    def wrapper(*args, **kwargs):
        print(f"   [calling {func.__name__} -- happens on every call]")
        return func(*args, **kwargs)
    return wrapper


print("about to define the function...")

@noisy_decorator
def ping():
    return "pong"

print("function defined; nothing has been called yet")

about to define the function...
   [decorating ping -- happens once, at definition]
function defined; nothing has been called yet


In [17]:
print(ping())
print(ping())

   [calling ping -- happens on every call]
pong
   [calling ping -- happens on every call]
pong


The `[decorating ...]` line appeared while the cell defining `ping` was running, and never again. This is why an expensive setup step belongs in the decorator body rather than inside `wrapper` — it is paid once instead of on every call.

---
# 4. Making a Decorator Work on Any Function

Our timer has a hidden limitation. `wrapper(data)` accepts exactly one argument, so the moment we decorate a function with a different signature it breaks:

In [18]:
@execution_timer
def upload_file(filename, bucket):
    time.sleep(0.2)


try:
    upload_file("report.csv", "analytics-bucket")
except TypeError as e:
    print("TypeError:", e)

TypeError: execution_timer.<locals>.wrapper() takes 1 positional argument but 2 were given


A decorator cannot know in advance what the functions it wraps will look like. It needs to accept **anything** and pass it straight through — which is what `*args` and `**kwargs` are for.

## `*args` (a recap) and `**kwargs` (new)

Chapter 6 introduced `*args`: it collects any number of **positional** arguments into a tuple.

`**kwargs` is its partner: it collects any number of **keyword** arguments into a dictionary. The names are conventions, not rules — it is the `*` and `**` that do the work.

In [19]:
def describe(**details):
    print(type(details))
    for key, value in details.items():
        print(f"  {key}: {value}")


describe(title="Dune", year=2021, rating=8.0)

<class 'dict'>
  title: Dune
  year: 2021
  rating: 8.0


Used together, a function can accept literally any call:

In [20]:
def log_event(event, *args, **kwargs):
    print("event:           ", event)
    print("extra positional:", args)        # a tuple
    print("extra keyword:   ", kwargs)      # a dict


log_event("purchase", 499, 2, user="amit", currency="INR")

event:            purchase
extra positional: (499, 2)
extra keyword:    {'user': 'amit', 'currency': 'INR'}


The `*` and `**` work in the other direction too. On the way **in** to a definition they *collect*; at a **call site** they *spread out*:

In [21]:
settings = {"user": "sara", "currency": "USD"}
extras = [99, 1]

log_event("refund", *extras, **settings)    # unpack the list and the dict into arguments

event:            refund
extra positional: (99, 1)
extra keyword:    {'user': 'sara', 'currency': 'USD'}


## The Universal Wrapper

Put those together and you get the shape that almost every decorator uses:

In [22]:
def execution_timer(func):
    def wrapper(*args, **kwargs):                   # accept anything
        start = time.perf_counter()
        result = func(*args, **kwargs)              # pass it all straight through
        duration = time.perf_counter() - start
        print(f"AWS: billed for {duration:.1f} seconds")
        return result                               # and give back what the function returned
    return wrapper

In [23]:
@execution_timer
def upload_file(filename, bucket):
    time.sleep(0.2)
    return f"uploaded {filename} to {bucket}"


print(upload_file("report.csv", bucket="analytics-bucket"))

AWS: billed for 0.2 seconds
uploaded report.csv to analytics-bucket


Two arguments now, one positional and one keyword, and it works.

**Note that `return result` line.** It is the most common decorator bug there is. Leave it out and the wrapper returns `None`, so every decorated function silently loses its return value:

In [24]:
def broken_timer(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)           # runs the function but throws the answer away
    return wrapper


@broken_timer
def add(a, b):
    return a + b


print(add(2, 3))                        # None, not 5 -- and nothing warned us

None


---
## The Identity Problem, and `functools.wraps`

A decorated function has quietly become a different object. Ask it its name:

In [25]:
@execution_timer
def generate_report(month):
    """Build the monthly billing report."""
    return f"report for {month}"


print(generate_report.__name__)
print(generate_report.__doc__)

wrapper
None


It calls itself `wrapper`, because that is literally what it now is. The original function is hidden inside the closure.

This is not cosmetic. Debuggers, error messages, documentation tools and test frameworks all read `__name__` and `__doc__`, and every decorated function in your project claiming to be `wrapper` makes a traceback much harder to read.

`functools.wraps` fixes it — itself a decorator, applied to the wrapper:

In [26]:
import functools

def execution_timer(func):
    @functools.wraps(func)                      # copy the original's identity onto wrapper
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"AWS: billed for {time.perf_counter() - start:.1f} seconds")
        return result
    return wrapper

In [27]:
@execution_timer
def generate_report(month):
    """Build the monthly billing report."""
    return f"report for {month}"


print(generate_report.__name__)
print(generate_report.__doc__)

generate_report
Build the monthly billing report.


Same behaviour, correct identity. **Use `@functools.wraps` in every decorator you write.** It costs one line and there is no reason to leave it out.

---
# 5. Decorators You Will Actually Meet

You will write decorators occasionally. You will *use* other people's constantly. These are the ones worth recognising on sight.

## `@property` — Methods That Look Like Attributes

Chapter 8 gave the `Account` class a private `__balance` with `get_balance()` and `set_balance()` methods, because reading and writing it directly would skip validation. It worked, but callers had to write `account.get_balance()` for something that feels like a plain attribute.

`@property` gives you both: attribute syntax, method behaviour.

In [28]:
class Account:
    def __init__(self, name, balance=0):
        self.name = name
        self._balance = balance

    @property
    def balance(self):
        # Runs when someone READS account.balance
        return self._balance

    @balance.setter
    def balance(self, amount):
        # Runs when someone WRITES account.balance = ...
        if amount < 0:
            raise ValueError("balance cannot be negative")
        self._balance = amount

In [29]:
acc = Account("Shikhar", 5000)

print(acc.balance)              # no parentheses -- but the getter ran

acc.balance = 7500              # looks like assignment -- but the setter ran
print(acc.balance)

try:
    acc.balance = -100          # and the validation still applies
except ValueError as e:
    print("ValueError:", e)

5000
7500
ValueError: balance cannot be negative


**One deliberate change from chapter 8.** That class stored `self.__balance` with two underscores; this one uses `self._balance` with one. The reason is that the property is now called `balance`, so the stored value needs a different name — and the single underscore is the convention for "internal, please leave this alone".

Two underscores would still work, but chapter 8's name mangling (`__balance` becoming `_Account__balance`) makes debugging and subclassing more awkward, and the property is already doing the job the double underscore was there for: nobody can set an invalid balance, because every write goes through the setter.

Compare that with chapter 8's version. The class kept full control, and the caller writes plain, readable code. This is the standard way to do encapsulation in modern Python.

A property can also be **computed**, with no stored value behind it at all:

In [30]:
class Movie:
    def __init__(self, title, runtime_minutes):
        self.title = title
        self.runtime_minutes = runtime_minutes

    @property
    def runtime_hours(self):
        return round(self.runtime_minutes / 60, 1)


dune = Movie("Dune", 155)
print(f"{dune.title}: {dune.runtime_minutes} min = {dune.runtime_hours} hours")

dune.runtime_minutes = 166      # change the source...
print("after the extended cut:", dune.runtime_hours)

Dune: 155 min = 2.6 hours
after the extended cut: 2.8


`runtime_hours` is never stored, so it can never fall out of sync with `runtime_minutes`.

## `@staticmethod` and `@classmethod`

Chapter 8's methods all took `self`, because they worked on one particular object. Two decorators change what a method receives:

- **`@staticmethod`** — no `self` at all. A plain function that lives inside the class because it belongs there logically.
- **`@classmethod`** — receives the **class** (`cls`) rather than an instance. Mostly used to build alternative constructors.

In [31]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @staticmethod
    def is_freezing(celsius):
        # No self: this does not depend on any particular Temperature object.
        return celsius <= 0

    @classmethod
    def from_fahrenheit(cls, f):
        # cls is the Temperature class -- so this builds and returns an instance.
        return cls(round((f - 32) * 5 / 9, 1))


print(Temperature.is_freezing(-4))              # called on the class, no object needed

body = Temperature.from_fahrenheit(98.6)        # an alternative constructor
print(f"{body.celsius}°C")

True
37.0°C


## `@functools.lru_cache` — Remember Previous Results

Some functions are called repeatedly with the same arguments. `lru_cache` stores what they returned and skips the work the second time.

Chapter 7's recursive Fibonacci is the classic demonstration, because it recomputes the same values an absurd number of times. Let us count the calls:

In [32]:
calls = 0

def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)


fib(25)
print(f"without a cache: {calls:,} calls")

without a cache: 242,785 calls


In [33]:
calls = 0

@functools.lru_cache
def fib_cached(n):
    global calls
    calls += 1
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)


fib_cached(25)
print(f"with a cache:    {calls:,} calls")

with a cache:    26 calls


Same function, same answer, one decorator. The cache turns an exponential amount of work into a linear amount.

You can ask the cache how it did:

In [34]:
print(fib_cached.cache_info())

CacheInfo(hits=23, misses=26, maxsize=128, currsize=26)


`hits` is how often an answer was reused, `misses` how often the work was actually done. By default the cache keeps the last 128 distinct calls; pass `@functools.lru_cache(maxsize=1000)` to change that, or `@functools.cache` (Python 3.9+) for a shorthand meaning "no limit".

**The catch:** an unbounded cache lives for the life of your program and grows as you call the function with new arguments, and every argument must be hashable — so no lists or dicts. Use it for pure calculations that always give the same answer for the same input, never for anything that reads changing data such as a file or an API.

## `@abstractmethod` — The Chapter 8 Loose End

Chapter 8 built a payment-gateway plugin architecture and wanted to force every subclass to implement `pay()`. Without decorators, the best we could do was raise `NotImplementedError` — which only complains *when the missing method is called*, possibly long after the broken class was written.

`@abstractmethod` refuses to let the incomplete class be created at all:

In [35]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def pay(self, amount):
        ...

class UPIGateway(PaymentGateway):
    def pay(self, amount):
        return f"Paid ₹{amount} via UPI"

class BrokenGateway(PaymentGateway):
    pass                                # forgot to implement pay()

In [36]:
print(UPIGateway().pay(499))

try:
    BrokenGateway()
except TypeError:
    print("TypeError: cannot create BrokenGateway -- pay() has not been implemented")

Paid ₹499 via UPI
TypeError: cannot create BrokenGateway -- pay() has not been implemented


The error arrives the moment someone tries to use the incomplete class, not later at some random call site. That is the tool chapter 8 was pointing at.

## Decorating Methods

Your own decorators work on methods too, with no changes — a method is just a function whose first argument happens to be `self`, and `*args` picks it up like any other:

In [37]:
class PlaylistService:
    @execution_timer
    def rebuild(self, name):
        time.sleep(0.2)
        return f"rebuilt {name}"


print(PlaylistService().rebuild("Road Trip"))

AWS: billed for 0.2 seconds
rebuilt Road Trip


`self` arrives inside `args` as the first element and is passed straight through. Nothing special is required.

## When Not to Reach for a Decorator

Decorators add a layer of indirection, and that has a cost. Skip them when:

- **It happens once.** If only one function needs the behaviour, just write the behaviour into that function.
- **The wrapper needs to know too much.** If your decorator starts inspecting argument names or branching on which function it wrapped, it wants to be a normal function instead.
- **The behaviour is the point.** A decorator should add a *policy* — timing, retrying, caching, logging. If it changes what the function fundamentally does, hiding that above the `def` makes the code harder to read, not easier.

The test: someone reading the `def` and ignoring the `@` lines should still understand what the function does.

---
# 6. Decorators That Take Arguments

*This section is a step up. The decorators above cover most day-to-day use — come back to this one when you need it.*

Sometimes a decorator needs configuring: retry **how many** times, cache **how many** results. Since `@something(...)` is a call, `something(...)` has to **return a decorator** — which means three nested functions:

```
repeat(3)          ->  returns decorator
decorator(func)    ->  returns wrapper
wrapper(*args)     ->  runs the function
```

In [38]:
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(3)
def send_reminder(email):
    print(f"   reminder sent to {email}")


send_reminder("user@example.com")

   reminder sent to user@example.com
   reminder sent to user@example.com
   reminder sent to user@example.com


Read the layers outside-in: `repeat(3)` runs first and returns `decorator`; then `decorator(send_reminder)` runs and returns `wrapper`; then `send_reminder` is that `wrapper`.

It is the same `func = decorator(func)` rule, with one extra step in front.

## The Mistake This Sets Up

Because `@repeat(3)` is a call and `@execution_timer` is not, it is easy to write one when you meant the other. Forget the `(3)` and the error is genuinely baffling the first time:

In [39]:
@repeat                          # oops -- no (3)
def announce():
    print("announcement sent")


try:
    announce()
except TypeError as e:
    print("TypeError:", e)

TypeError: repeat.<locals>.decorator() missing 1 required positional argument: 'func'


Work it through with the rule and it makes sense. `@repeat` means `announce = repeat(announce)` — so `announce` was passed in as `times`, and `repeat` handed back `decorator`, which is now sitting in the name `announce`. Calling `announce()` calls `decorator()` with no function to decorate.

The same mistake in reverse — writing `@execution_timer()` for a decorator that takes no arguments — fails just as strangely, because `execution_timer()` is called with no function at all.

**The rule:** parentheses only if the decorator takes configuration. When you hit a confusing `TypeError` on a decorated function, this is the first thing to check.

---
# Real-World Example: Retrying a Flaky Call

Network calls fail for no good reason and succeed when you try again. Wrapping every call in retry logic by hand is repetitive; a decorator does it once.

You will reuse this exact idea in chapter 17, on a real API.

In [40]:
def retry(attempts=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as e:
                    print(f"   attempt {attempt} failed: {e}")
            raise RuntimeError(f"gave up after {attempts} attempts")
        return wrapper
    return decorator

In [41]:
state = {"calls": 0}

@retry(attempts=3)
def fetch_user(user_id):
    # Fails the first two times, then succeeds -- standing in for a flaky network.
    state["calls"] += 1
    if state["calls"] < 3:
        raise ConnectionError("network unreachable")
    return {"id": user_id, "name": "amit"}


print(fetch_user(42))

   attempt 1 failed: network unreachable
   attempt 2 failed: network unreachable
{'id': 42, 'name': 'amit'}


`fetch_user` itself contains no retry logic at all. It describes what it fetches; the decorator describes how failure is handled. Adding retries to another function is now one line.

That separation — **the function does the work, the decorator adds the policy** — is what decorators are for. Logging, timing, caching, access checks, rate limiting and retries are all policies, and all of them show up as decorators in real codebases.

## Stacking Decorators

You can apply more than one. They are applied **bottom-up**: the one nearest the `def` wraps the function first.

In [42]:
@execution_timer          # applied second, so it wraps the retrying version
@retry(attempts=3)        # applied first, closest to the function
def sync_playlist(name):
    time.sleep(0.2)
    return f"synced {name}"


print(sync_playlist("Road Trip"))

AWS: billed for 0.2 seconds
synced Road Trip


Reading order matters: `@execution_timer` on top means the timer measures *all* the retry attempts together, not each one separately. Swap the two lines and you would time each attempt instead. Neither is wrong, but they answer different questions.

---
# Quick Reference

**The one rule**

```python
@decorator
def func():          #  is exactly the same as writing
    ...              #  func = decorator(func)
```

**The decorator template — start from this every time**

```python
import functools

def my_decorator(func):
    @functools.wraps(func)              # keeps func's name and docstring
    def wrapper(*args, **kwargs):       # accepts any signature
        # ... before ...
        result = func(*args, **kwargs)  # pass everything through
        # ... after ...
        return result                   # never forget this
    return wrapper
```

**Collecting and spreading**

| Where | `*` | `**` |
|---|---|---|
| In a `def` | collects extra positional args into a tuple | collects extra keyword args into a dict |
| At a call | spreads a list/tuple into positional args | spreads a dict into keyword args |

**Closures**

```python
def make(x):
    def inner(y):
        return x + y     # remembers x after make() has returned
    return inner
```

| Keyword | Rebinds a name in |
|---|---|
| `nonlocal` | the enclosing function |
| `global` | the module |

**Built-in decorators worth knowing**

| Decorator | Purpose |
|---|---|
| `@property` | Method that reads like an attribute |
| `@x.setter` | The matching writer, with validation |
| `@staticmethod` | Method with no `self` |
| `@classmethod` | Method that receives the class as `cls` |
| `@functools.wraps` | Preserve the wrapped function's identity |
| `@functools.lru_cache` | Cache results of a pure function |
| `@abstractmethod` | Subclass must implement this |

**The three classic bugs**

1. Forgetting `return result` in the wrapper — the function silently returns `None`.
2. Forgetting `@functools.wraps` — every function ends up named `wrapper`.
3. Writing `@decorator()` when the decorator takes no arguments, or `@decorator` when it does.

---

**Next:** chapter 14 covers context managers — what the `with` statement you have used since chapter 10 is actually doing, and how to write your own using the generators and decorators from these two chapters.